In [1]:
from langchain_groq import ChatGroq
import os
from load_dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

if os.environ.get("GROQ_API_KEY"):
    print("API KEY FETCHED")
else:
    raise ValueError("GROQ_API_KEY Not Found!!")

llm_groq = init_chat_model(model = "llama-3.1-8b-instant", model_provider="groq")

llm_groq.invoke("Hello!")

API KEY FETCHED


AIMessage(content='Hello. How can I assist you today?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 37, 'total_tokens': 47, 'completion_time': 0.0110111, 'completion_tokens_details': None, 'prompt_time': 0.003094984, 'prompt_tokens_details': None, 'queue_time': 0.050875445, 'total_time': 0.014106084}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d42d5-5cfd-7ab0-a853-eb2a2c3e41c5-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 37, 'output_tokens': 10, 'total_tokens': 47})

# **CHAIN WITH PARALLEL CHAINS**

In [2]:
# Task 1 - Prompt

from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a movie summarizer"),
    ("human", "Please summarize the movie in brief : {input}")
])

In [3]:
# Task 2 - LLM

llm_groq = init_chat_model(model = "llama-3.1-8b-instant", model_provider="groq")

In [4]:
# Task 3 - String Parser

str_parser = StrOutputParser()

In [6]:
# Task 4 - Custom Runnable

from langchain_core.runnables import RunnableLambda

def dictionary_maker(text:str)->dict:

    return {"text" : text}

dictionary_maker_runnable = RunnableLambda(dictionary_maker)

# **PARALLEL CHAIN 1**

In [7]:
# Task 1 - Prompt

linkedin_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a LinkedIn post generator"),
    ("human", "Create a post for the following text for Linkedin: {text}")
])

# Task 2 - LLM

llm_groq = init_chat_model(model = "llama-3.1-8b-instant", model_provider="groq")

# Task 3 - String Parser

str_parser = StrOutputParser()

chain_linkedin = linkedin_prompt | llm_groq | str_parser

# **PARALLEL CHAIN 2**

In [17]:
from langchain_core.runnables import RunnableParallel, RunnableLambda

In [18]:
def insta_chain(text:dict):

    text : text["text"]

    insta_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a Instagram post generator"),
        ("human", "Create a post for the following text for Instagram: {text}")
    ])

    # Task 2 - LLM

    llm_groq = init_chat_model(model = "llama-3.1-8b-instant", model_provider="groq")

    # Task 3 - String Parser

    str_parser = StrOutputParser()

    chain_insta = insta_prompt | llm_groq | str_parser

    result = chain_insta.invoke(text)

    return result

insta_chain_runnable = RunnableLambda(insta_chain)

# **Final Orchestration**

In [19]:
final_chain = (
                prompt_template | 
                llm_groq |
                str_parser |
                dictionary_maker_runnable | 
                RunnableParallel (branches = {"LinkedIn" : chain_linkedin, "Instagram" : insta_chain_runnable})
)

In [20]:
final_chain.invoke("KGF")

{'branches': {'LinkedIn': 'Here\'s a LinkedIn-style post based on the text:\n\n**Title:** "Empowerment Through Courage: Lessons from KGF"\n\n**Text:**\n\nAs leaders, we often face challenges that seem insurmountable. But it\'s our ability to rise above adversity and fight for what\'s right that truly defines our character.\n\nI\'m reflecting on the inspiring story of Rocky Yadav from the film KGF, a young boy from Mumbai who transforms into a powerful force for justice in the Kolar Gold Fields. His journey is a testament to the power of resilience, determination, and a unwavering commitment to one\'s values.\n\nAs we navigate our own professional journeys, let\'s draw inspiration from Rocky\'s courage and conviction. Remember that we all have the power to create change and fight against injustice.\n\n**Key Takeaways:**\n\n- Courage is not the absence of fear, but the willingness to act in the face of adversity.\n- Our values and principles are what guide us towards making a positive im

# **CHAIN AS A RUNNABLE**

In [27]:
# Task 1 - Beautify

def beautify(final_response:dict)->dict:

    linkedin_response = final_response['branches']["LinkedIn"]
    instagram_response = final_response['branches']["Instagram"]

    return {"LinkedIn" : linkedin_response, "Instagram" : instagram_response}

beautify_runnable = RunnableLambda(beautify)

# Task 2 - Final Chain

beautified_chain = final_chain | beautify_runnable

beautified_chain.invoke("KGF")

{'LinkedIn': "Here's a potential LinkedIn post based on the text:\n\n**Lessons from the Underdog**\n\nAs I reflect on the inspiring story of Rocky from the movie KGF, I'm reminded of the power of resilience and determination.\n\nRocky's journey from a poor and struggling miner's son to becoming one of the most powerful men in the region is a testament to the fact that success is not a destination, but a journey.\n\nHere are a few key takeaways from Rocky's story:\n\n **Ambition knows no bounds**: Despite facing numerous challenges and setbacks, Rocky never lost sight of his goal to become the richest man in India.\n\n **Perseverance pays off**: Rocky's unwavering commitment to his mission ultimately led him to achieve success, even in the face of adversity.\n\n **Leadership is about serving others**: As Rocky rose to power, he didn't forget where he came from and remained true to his roots, serving the people who had supported him along the way.\n\nWhat can we learn from Rocky's story?